<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/SBERT_chatbot_260513.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BERT의 문장 임베딩을 이용한 한국어 챗봇 만들기


In [ ]:
!pip install -U sentence-transformers

  Using cached sentence_transformers-5.5.0-py3-none-any.whl.metadata (18 kB)
Using cached sentence_transformers-5.5.0-py3-none-any.whl (588 kB)
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 2.2.2
    Uninstalling sentence-transformers-2.2.2:
      Successfully uninstalled sentence-transformers-2.2.2


GPU 확인하기

In [ ]:
import torch

print(torch.cuda.is_available())

True


모델 로드

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('jhgan/ko-sroberta-multitask')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

문장 임베딩 테스트

In [ ]:
sentences = ['안녕하세요', '오늘 기분이 좋아요']

embeddings = model.encode(sentences)

print(embeddings.shape)

(2, 768)


문장 2개를 각각 숫자 768개짜리 벡터로 바꿨다는 뜻.

train_data의 각 질문(Q)을 하나씩 꺼내서 BERT 모델로 숫자 벡터로 바꾸고

그 결과를 embedding이라는 새 열(column)에 저장하기


파일 다운로드

In [ ]:
!wget https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv

--2026-05-13 14:21:18--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv’

ChatbotData.csv     100%[===================>] 868.99K  --.-KB/s    in 0.006s  

2026-05-13 14:21:18 (151 MB/s) - ‘ChatbotData.csv’ saved [889842/889842]



In [ ]:
import pandas as pd

train_data = pd.read_csv("ChatbotData.csv")
train_data.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [ ]:
# 데이터 개수 확인
print(len(train_data))
print(train_data.columns)

11823
Index(['Q', 'A', 'label'], dtype='object')


In [ ]:
# 임베딩 생성

train_data['embedding'] = train_data['Q'].apply(lambda x: model.encode(x))

In [ ]:
# 임베딩 확인
train_data.head()

,Q,A,label,embedding
0,12시 땡!,하루가 또 가네요.,0,"[-0.19072695, 0.3214757, 0.59873754, 0.4461786..."
1,1지망 학교 떨어졌어,위로해 드립니다.,0,"[0.12320574, 0.6423682, 0.6853603, -0.42912346..."
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0,"[0.024357503, -0.22393543, 0.22917706, -0.4501..."
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0,"[0.036285225, -0.21048254, 0.07269403, -0.5403..."
4,PPL 심하네,눈살이 찌푸려지죠.,0,"[0.022030605, 0.7055219, -0.17157203, 0.165971..."


In [ ]:
from numpy import dot
from numpy.linalg import norm
import numpy as np

코사인 유사도 함수 : 두 문장이 얼마나 비슷한 방향인지 점수 계산

In [ ]:
def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

In [ ]:
def return_answer(question):

    embedding = model.encode(question)

    train_data['score'] = train_data['embedding'].apply(
        lambda x: cos_sim(x, embedding)
    )

    return train_data.loc[train_data['score'].idxmax()]['A']

In [ ]:
return_answer("배고파")

'얼른 맛난 음식 드세요.'

In [ ]:
# 확실하게 보려면
print(return_answer("배고파"))

얼른 맛난 음식 드세요.


실제 챗봇처럼 여러번 질문하기

In [ ]:
while True:

    question = input("나: ")

    if question == "종료":
        break

    answer = return_answer(question)

    print("챗봇:", answer)

나: 지금 몇시니
챗봇: 시간은 지금도 흐르고 있어요.
나: 종료


사용자가 질문 입력하면 질문을 BERT 숫자벡터로 변환한다

기존 질문들과 비교하여 가장 비슷한 질문의 답변 출력한다

- 생성이 아닌 가장 비슷한 질문 찾아서 답변 꺼내오기 방식

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
import os
# 내 드라이브에서 ipynb 찾기
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.ipynb'):
            print(os.path.join(root, file))

/content/drive/MyDrive/Colab Notebooks/today_260127.ipynb
/content/drive/MyDrive/Colab Notebooks/today_260126.ipynb
/content/drive/MyDrive/Colab Notebooks/today_260129.ipynb
/content/drive/MyDrive/Colab Notebooks/today_260131.ipynb
/content/drive/MyDrive/Colab Notebooks/python2_260131.ipynb
/content/drive/MyDrive/Colab Notebooks/wordcloud_260131.ipynb
/content/drive/MyDrive/Colab Notebooks/plotly_260131.ipynb
/content/drive/MyDrive/Colab Notebooks/plotly_260202.ipynb
/content/drive/MyDrive/Colab Notebooks/plotly_260203.ipynb
/content/drive/MyDrive/Colab Notebooks/plotly_260204.ipynb
/content/drive/MyDrive/Colab Notebooks/plotly2_260204.ipynb
/content/drive/MyDrive/Colab Notebooks/plotly_260205.ipynb
/content/drive/MyDrive/Colab Notebooks/book_260223.ipynb
/content/drive/MyDrive/Colab Notebooks/spotify_project.ipynb
/content/drive/MyDrive/Colab Notebooks/LinearAlgebra_RREF.ipynb
/content/drive/MyDrive/Colab Notebooks/graph260314.ipynb
/content/drive/MyDrive/Colab Notebooks/m260316.ipynb

In [31]:
# 코랩에서 실행
!pip install nbformat

import nbformat

with open("/content/drive/MyDrive/Colab Notebooks/SBERT_chatbot_260513.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

# 위젯 메타데이터 제거
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

with open("/content/drive/MyDrive/Colab Notebooks/SBERT_chatbot_260513.ipynb", "w") as f:
    nbformat.write(nb, f)

print("완료!")

완료!
